In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")
%matplotlib inline

# --- Load data ---
df = pd.read_csv('insurance_claims.csv')

if '_c39' in df.columns:
    df = df.drop(columns=['_c39'])

# --- Clean missing-value placeholders ---
df = df.replace('?', np.nan)

# --- Encode fraud target ---
df['fraud_label'] = df['fraud_reported'].map({'Y': 1, 'N': 0})

print("Loaded shape:", df.shape)

# ============================================================
# FEATURE ENGINEERING
# ============================================================

# --- 1. Drop identifier / non-predictive columns ---
drop_cols = ['policy_number', 'policy_bind_date', 'insured_zip', 'incident_date',
             'incident_location', 'auto_model']

df_fe = df.drop(columns=[c for c in drop_cols if c in df.columns])

# --- 2. Fill structurally missing categoricals with explicit 'Unknown' ---
for col in ['collision_type', 'authorities_contacted', 'property_damage', 'police_report_available']:
    df_fe[col] = df_fe[col].fillna('Unknown')

# --- 3. Engineered features confirmed by EDA ---
df_fe['multi_vehicle_incident'] = (df_fe['number_of_vehicles_involved'] >= 2).astype(int)

df_fe['severity_vehicle_interaction'] = (
    df_fe['incident_severity'].astype(str) + "_" + df_fe['multi_vehicle_incident'].astype(str)
)

print("Shape after cleanup:", df_fe.shape)
print("\nNew columns check:")
print(df_fe[['multi_vehicle_incident', 'severity_vehicle_interaction']].head())
print("\nValue counts - severity_vehicle_interaction:")
print(df_fe['severity_vehicle_interaction'].value_counts())

# --- 4. Leakage-safe feature set for SEVERITY model ---
# injury_claim/property_claim/vehicle_claim sum to total_claim_amount - must exclude
severity_leakage_cols = ['injury_claim', 'property_claim', 'vehicle_claim',
                          'fraud_reported', 'fraud_label', 'total_claim_amount']
X_severity = df_fe.drop(columns=[c for c in severity_leakage_cols if c in df_fe.columns])
y_severity = df_fe['total_claim_amount']

print("\nX_severity shape:", X_severity.shape)
print("X_severity columns:", X_severity.columns.tolist())

# --- 5. Feature set for FRAUD model (claim components ARE allowed here) ---
fraud_leakage_cols = ['fraud_reported', 'fraud_label']
X_fraud = df_fe.drop(columns=[c for c in fraud_leakage_cols if c in df_fe.columns])
y_fraud = df_fe['fraud_label']

print("\nX_fraud shape:", X_fraud.shape)
print("X_fraud columns:", X_fraud.columns.tolist())

# --- 6. Check remaining categorical cardinality before encoding ---
print("\nCategorical cardinality in X_severity:")
for col in X_severity.select_dtypes(include='object').columns:
    print(f"  {col}: {X_severity[col].nunique()}")

Loaded shape: (1000, 40)
Shape after cleanup: (1000, 36)

New columns check:
   multi_vehicle_incident severity_vehicle_interaction
0                       0               Major Damage_0
1                       0               Minor Damage_0
2                       1               Minor Damage_1
3                       0               Major Damage_0
4                       0               Minor Damage_0

Value counts - severity_vehicle_interaction:
severity_vehicle_interaction
Minor Damage_0      224
Total Loss_1        145
Major Damage_1      144
Total Loss_0        135
Major Damage_0      132
Minor Damage_1      130
Trivial Damage_0     90
Name: count, dtype: int64

X_severity shape: (1000, 30)
X_severity columns: ['months_as_customer', 'age', 'policy_state', 'policy_csl', 'policy_deductable', 'policy_annual_premium', 'umbrella_limit', 'insured_sex', 'insured_education_level', 'insured_occupation', 'insured_hobbies', 'insured_relationship', 'capital-gains', 'capital-loss', 'incident_